# Fine-tunning on Figure 3D MethylBERT data with Refined MethylBERT

This tutorial demonstrates:
1. **How to perform fine-tunning on the prepared data and save results to the local directory**.
---

### Step 0: Import needed classes and set paths

In [1]:
import os
from methyldl.modelling.methylbert import MethylVocab,MethylBertFinetuneDataset
import pandas as pd
from methyldl.data.genome import generate_kmer_str_with_overlap

x:\KULeuven-Masters\Master Thesis\methyldl\methyldl\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from methyldl.modelling.methylbert import MethylBert,default_methylbert_config

In [3]:
data_path =  "../Data/Curated/rrms_dmrs_only_mincov40_corrected_top3000_dmr_cuts/chr1"

In [4]:
dmrs = pd.DataFrame(enumerate(pd.concat([pd.read_parquet(data_path+f"/{split}.parquet") for split in ["train", "valid", "test"]])["dmr_name"].unique()))
dmrs.columns  = ["dmr_id", "dmr_name"]

In [5]:
seq_length = 150

In [6]:
def prepare_methylbert_list(data_path, split,dmrs,seq_length = 500):
    data = pd.read_parquet(data_path+f"/{split}.parquet")
    data = data.merge(dmrs, on=["dmr_name"])
    data_list = [['dna_seq', 'methyl_seq', 'dmr_ctype', 'dmr_label','ctype']]
    for i,row in data.iterrows():
        dna = generate_kmer_str_with_overlap(row["input_ids"][:(seq_length+2)])
        methyl = row["methylation_ids"][1:-1][:seq_length]
        if not len(dna) or not len(methyl):
            # print(row["input_ids"])
            # print(row["read_name"])
            continue
        label = row["label"]
        dmr_label = row["dmr_id"]
        data_list.append([dna, methyl, 1,dmr_label,label])
    return data_list

In [7]:
train, valid, test = [prepare_methylbert_list(data_path, split=x,dmrs=dmrs, seq_length=seq_length) for x in ["train", "valid", "test"]]

In [8]:
train_dataset, valid_dataset, test_dataset = [MethylBertFinetuneDataset(data_source=x,vocab=MethylVocab(k=3),seq_len=seq_length) for x in [train, valid, test]]

Building Vocab
['dna_seq', 'methyl_seq', 'dmr_ctype', 'dmr_label', 'ctype']
Total number of sequences :  285526
# of reads in each label:  [116757 168769]
Building Vocab
['dna_seq', 'methyl_seq', 'dmr_ctype', 'dmr_label', 'ctype']
Total number of sequences :  88073
# of reads in each label:  [36393 51680]
Building Vocab
['dna_seq', 'methyl_seq', 'dmr_ctype', 'dmr_label', 'ctype']
Total number of sequences :  101847
# of reads in each label:  [39546 62301]


### Step 1: Perform fine-tunning

In [9]:
import torch

In [10]:
torch.__version__

'2.7.1+cu128'

In [11]:
from collections import OrderedDict

In [12]:
rrms_config = OrderedDict([
    ("lr", 0.0004),
    ("beta", (0.9, 0.98)),
    ("weight_decay", 0.1),
    ("warmup_step", 100),
    ("eps", 1e-6),
    ("with_cuda", True),
    ("log_freq", 20),
    ("eval_freq", 20),
    ("n_hidden", None),
    ("decrease_steps", 200),
    ("eval", False),
    ("amp", True),
    ("gradient_accumulation_steps", 1),
    ("max_grad_norm", 1.0),
    ("save_freq", None),
    ("loss", "bce"),
    ("adam_beta1", 0.9),
    ("adam_beta2", 0.98),
    ("seed", 950410),
])

In [ ]:
model_instance = MethylBert(custom_config=rrms_config, 
                            foundation_model_path = "hanyangii/methylbert_hg19_12l",
                            load_weights = True,
                            num_labels=2,
                            num_dmr_labels = len(dmrs),
                            seq_len=seq_length,
                            output_dir="../../fine_tunning_results/MethylBERT_RRMS/chr1_rrms_dmrs_only_mincov40_corrected_top3000_dmr_cuts/",
                            batch_size=500) 

In [ ]:
# model_instance.fine_tune(data_path=data_path)
model_instance.fine_tune(data_path=None, train_dataset=train_dataset, val_dataset=valid_dataset, test_dataset=test_dataset)

### Step 2: Test predictions

In [15]:
model_instance = MethylBert(custom_config=rrms_config, 
                            foundation_model_path = "hanyangii/methylbert_hg19_12l",
                            num_labels=2,
                            num_dmr_labels = len(dmrs),
                            fine_tuned_model_path="../../fine_tunning_results/MethylBERT_RRMS/chr1_rrms_dmrs_only_mincov40_corrected_top3000_dmr_cuts/checkpoint-200/model.safetensors") 

Loading MethylBertEmbeddedDMR weights from fine_tuned_model_path: ../../fine_tunning_results/MethylBERT_RRMS/chr1_rrms_dmrs_only_mincov40_corrected_top3000_dmr_cuts/checkpoint-200/model.safetensors
Cross entropy loss assigned


In [16]:
from methyldl.data.dataset import generate_example_data_for_methylbert

In [17]:
sequence_length = 150
synthetic_data = generate_example_data_for_methylbert(
                    sequence_length=sequence_length,
                    include_cpg_methylation=True,
                    include_m6a_methylation=False,
                    include_labels=True,
                    num_samples=2  # Single sample per repeat
                )

In [18]:
synthetic_data

[['dna_seq', 'methyl_seq', 'ctype'],
 ['AAG AGA GAA AAT ATG TGA GAA AAT ATG TGG GGT GTG TGG GGC GCG CGT GTG TGC GCC CCA CAA AAT ATG TGG GGC GCG CGG GGG GGA GAG AGA GAC ACG CGA GAA AAG AGG GGG GGC GCA CAG AGG GGG GGG GGT GTC TCT CTG TGC GCG CGT GTG TGC GCT CTC TCG CGA GAT ATG TGA GAG AGA GAC ACG CGC GCC CCC CCT CTC TCG CGC GCT CTA TAT ATG TGC GCC CCA CAT ATA TAC ACT CTC TCC CCC CCC CCG CGT GTT TTG TGA GAT ATA TAC ACA CAG AGG GGT GTT TTC TCC CCG CGT GTG TGA GAA AAT ATT TTA TAC ACT CTA TAC ACG CGT GTA TAG AGT GTT TTT TTC TCG CGC GCG CGA GAC ACC CCA CAC ACT CTG TGG GGA GAA AAG AGG GGC GCT CTA TAC ACG CGC GCT CTT TTC TCG CGT GTC TCG CGA',
  '222222222222222122201222222222222222222202222220221222020222222212100222022220122202021222222220222221122222222122022222222222212212222222022022222222',
  1],
 ['TTA TAA AAT ATC TCT CTG TGG GGC GCA CAC ACG CGG GGT GTT TTG TGA GAT ATC TCT CTC TCA CAA AAA AAG AGT GTT TTT TTA TAC ACC CCA CAC ACC CCT CTC TCT CTA TAA AAC ACT CTT TTT TTT TTA TAT ATA TAA AAC A

In [21]:
test_data = MethylBertFinetuneDataset(data_source=synthetic_data,
                                      vocab=MethylVocab(k=3),
                            seq_len=150)

Building Vocab
['dna_seq', 'methyl_seq', 'ctype']
Total number of sequences :  2
# of reads in each label:  [2]


In [20]:
# test_data = MethylBertFinetuneDataset(data_source=data_path+"/test.txt",
#                                       vocab=MethylVocab(k=3),
#                             seq_len=150)

In [22]:
res = model_instance.predict(test_data)

X:\KULeuven-Masters\Master Thesis\methyldl\methyldl\methyldl\modelling\methylbert.py:466: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
from sklearn.metrics import confusion_matrix, auc, roc_curve
import matplotlib.pyplot as plt
import numpy as np

# Set Matplotlib styling
plt.rcParams.update({
    "font.size": 12,
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "grid.color": "lightgray",
    "grid.linestyle": "-",
})

# Compute confusion matrix
cf_matrix = confusion_matrix(res.label_ids, res.predictions>0.5)

# Create the heatmap using Matplotlib
fig, ax = plt.subplots(1, figsize=(6, 6))
cax = ax.matshow(cf_matrix, cmap="PiYG")
plt.colorbar(cax)

# Annotate the heatmap
for (i, j), val in np.ndenumerate(cf_matrix):
    ax.text(j, i, f"{val}", ha="center", va="center", color="white" if val > cf_matrix.max() / 2 else "black")

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=12)
ax.set_ylabel("Ground-truth", fontsize=12)
ax.set_xticks([0, 1])
ax.set_xticklabels(["N", "T"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["N", "T"])

# Compute ROC and AUC
fpr, tpr, thresholds = roc_curve(res.label_ids, res.predictions)
auc_val = auc(fpr, tpr)

# Set title with AUC value
ax.set_title(f"MethylBERT test (auc: {auc_val:.3f})", fontsize=14)

# Show plot
plt.tight_layout()
plt.show()


In [ ]:
methylbert_formated_data = MethylBertFinetuneDataset(
                            data_source = synthetic_data,
                            vocab=MethylVocab(k=3),
                            seq_len=150
              )

In [ ]:
model_instance.predict(methylbert_formated_data)